# 使い方
**新しいColabを作る**
→ **GitHub上の標準ノートブックをColabで開く**
→ **そのノートブックを上から実行する**

では、そこからやり直します。

**Step 1: GitHub上のノートブックを開く**

Colab画面で、上部メニューから進んでください。

1. **File / ファイル**
2. **Open notebook / ノートブックを開く**
3. **GitHub** タブを選ぶ
4. 検索欄に以下を入れる

```text
KazuhTAKEH/-knowledge-state-reviewer
```

5. 一覧に出てくる以下を選ぶ

```text
notebooks/ksr_colab_starter.ipynb
```


# Knowledge State Reviewer - Colab Starter

This notebook is intentionally thin. Reusable logic should live in `src/ksr` so Codex can edit and test it locally.

In [1]:
# Optional: mount Drive when you need persistent data/checkpoints.
# from google.colab import drive
# drive.mount('/content/drive')

ファイルをオープンしただけだと、以下のようにgit cloneがコメントアウトされているので、コメントアウトを外す。
```python
# Clone or pull your repository here.
# !git clone https://github.com/KazuhTAKEH/-knowledge-state-reviewer.git /content/-knowledge-state-reviewer
%cd /content/-knowledge-state-reviewer
!pip install -q -r requirements-colab.txt
```

In [2]:
# Clone or pull your repository here.
!git clone https://github.com/KazuhTAKEH/-knowledge-state-reviewer.git /content/-knowledge-state-reviewer
%cd /content/-knowledge-state-reviewer
!pip install -q -r requirements-colab.txt

Cloning into '/content/-knowledge-state-reviewer'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 55 (delta 6), reused 55 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 20.05 KiB | 789.00 KiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/-knowledge-state-reviewer


In [3]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'src'))

from ksr.alignment import ConceptCandidate, LexicalConceptAligner, SentenceTransformerConceptAligner

concepts = [ConceptCandidate(**item) for item in json.loads(Path('examples/concepts/python_basic.json').read_text(encoding='utf-8'))]
phrases = ['defで関数を定義する', 'returnで結果を返す', '引数として値を受け取る']

aligner = LexicalConceptAligner()
aligner.align(phrases, concepts, threshold=0.25)

[AlignmentResult(input_phrase='defで関数を定義する', concept_id='function', label='関数定義 def', score=0.3529, method='lexical'),
 AlignmentResult(input_phrase='returnで結果を返す', concept_id='return_value', label='戻り値 return', score=0.4151, method='lexical')]

In [4]:
# Dependency-free smoke test. Run this before GPU/BERT experiments.
!PYTHONPATH=src PYTHONIOENCODING=utf-8 python scripts/colab_smoke_test.py

{
  "test": "lightweight",
  "artifact_concepts": [
    "function",
    "function_call",
    "return_value"
  ],
  "gap_concepts": [
    "function",
    "function_call",
    "return_value"
  ],
  "alignments": [
    {
      "input_phrase": "関数を定義する",
      "concept_id": "function",
      "label": "関数定義 def",
      "score": 0.2979,
      "method": "lexical"
    },
    {
      "input_phrase": "returnで結果を返す",
      "concept_id": "return_value",
      "label": "戻り値 return",
      "score": 0.4151,
      "method": "lexical"
    },
    {
      "input_phrase": "引数として値を受け取る",
      "concept_id": "function_call",
      "label": "関数呼び出しと引数 parameter argument",
      "score": 0.2222,
      "method": "lexical"
    }
  ],
  "ok": true
}


In [5]:
# BERT/SentenceTransformer semantic alignment. Use GPU runtime if available.
aligner = SentenceTransformerConceptAligner()
aligner.align(phrases, concepts, threshold=0.45)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[AlignmentResult(input_phrase='defで関数を定義する', concept_id='function', label='関数定義 def', score=0.8947, method='sentence-transformers:sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'),
 AlignmentResult(input_phrase='returnで結果を返す', concept_id='return_value', label='戻り値 return', score=0.6242, method='sentence-transformers:sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')]

In [6]:
# Run the current local CLI from Colab.
!PYTHONPATH=src PYTHONIOENCODING=utf-8 python -m ksr.cli dialogue --input examples/monologue.txt --profile examples/profiles/beginner_python.json

# Dialogue Draft

**先生:** Pythonの関数は、処理に名前を付けて再利用できるようにする仕組みです。 defで関数を定義し、必要な値を引数として受け取り、returnで結果を返します。 同じ処理を何度も書かずに済むので、プログラムを読みやすく保てます。

**生徒:** beginner_python の自分には、`function` が急に出てきた感じがします。何を表しているんですか？

**先生:** まず `function` が使われている箇所を一緒に探しましょう。その後、自分の言葉で一文にしてみてください。

**生徒:** `function_call` は何となく分かるのですが、この説明の中ではどこに効いていますか？

**先生:** まず `function_call` が使われている箇所を一緒に探しましょう。その後、自分の言葉で一文にしてみてください。

**生徒:** beginner_python の自分には、`return_value` が急に出てきた感じがします。何を表しているんですか？

**先生:** まず `return_value` が使われている箇所を一緒に探しましょう。その後、自分の言葉で一文にしてみてください。


## Teacher Explanation Review

- 説明は、生徒の弱い概念を先に確認してから進める必要があります。

- 教師役は完成文を渡すより、短い再記述課題を挟む方が自律修正につながります。

- `function` について、理解確認の問いを1つ追加してください。

- `function_call` について、理解確認の問いを1つ追加してください。

- `return_value` について、理解確認の問いを1つ追加してください。



In [7]:
!PYTHONPATH=src PYTHONIOENCODING=utf-8 python scripts/colab_smoke_test.py

{
  "test": "lightweight",
  "artifact_concepts": [
    "function",
    "function_call",
    "return_value"
  ],
  "gap_concepts": [
    "function",
    "function_call",
    "return_value"
  ],
  "alignments": [
    {
      "input_phrase": "関数を定義する",
      "concept_id": "function",
      "label": "関数定義 def",
      "score": 0.2979,
      "method": "lexical"
    },
    {
      "input_phrase": "returnで結果を返す",
      "concept_id": "return_value",
      "label": "戻り値 return",
      "score": 0.4151,
      "method": "lexical"
    },
    {
      "input_phrase": "引数として値を受け取る",
      "concept_id": "function_call",
      "label": "関数呼び出しと引数 parameter argument",
      "score": 0.2222,
      "method": "lexical"
    }
  ],
  "ok": true
}
